In [ ]:
import sys
import pandas as pd

import os
import json 
from pathlib import Path

# Access packages from the root
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..")))

from app.agent.rag.ingestion.data_ingestion import process_and_load_file
from app.agent.rag.ingestion.embeddings import get_embedding_model, generate_embeddings
from app.agent.rag.ingestion.vector_store import get_vector_store, add_documents_to_chroma
from app.agent.rag.chains.notes_chain import run_notes_chain
from app.agent.rag.retrieval.retriever import get_semantic_chunks, get_topic_chunks
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI


In [2]:
# Load .env variables
load_dotenv()
api_key = os.environ.get('GOOGLE_API_KEY')

print(os.getcwd())


c:\Users\USER\Documents\AI projects\DenseLess\app\agent


In [3]:
json_eval_dataset = {}
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    project="denseless",
    location="us-central1",
    vertexai=True,
)


INFO:google_genai._api_client:The user provided project/location will take precedence over the Vertex AI API key from the environment variable.


In [4]:
# Extract all pdf filenames
folder = Path("../pdfs")
files = [f.name for f in folder.iterdir() if f.is_file()]
print(files)

['ADDER CIRCUIT.pdf', 'BASIC OS CONCEPTS.pdf', 'Computer Circuit.pdf', 'CSC 315Week OneTTTT2.pdf', 'CSC315 Number System, Codes and Addressing Modes (2).pdf', 'Disaster Recovery Plan.pdf', 'History of Integrated Circuits video presentation slides (2).pdf', 'Incident Response.pdf', 'Interconnection Structures.pdf', 'Learning Memory System.pdf', 'MEMORY MANAGEMENT.pdf', 'PROCESS MANAGEMENT.pdf', 'Software Engineering Intro.pdf', 'Week 1- Fundamentals.pdf', 'Week 2- Algorithm Correctness and Mathematical Induction.pdf', 'Week 3- Algorithm Efficiency and Asymptotic notation.pdf', 'Week 4- Sorting Algorithms I.pdf']


In [ ]:
# Worked for first 14
# Generate each note and build eval dataset
i = 1
for file in files:
    # Run ingestion
    topic = file.removesuffix(".pdf")
    docs = process_and_load_file(f"../pdfs/{file}")
    embedder = get_embedding_model()
    vectors = generate_embeddings(docs, embedder)
    store = get_vector_store(student_id="1019", embedder=embedder)
    add_documents_to_chroma(store, vectors, docs, False, "CSU", topic, topic)

    # Run topic-wide retrieval
    chunks_topic = get_topic_chunks(store, topic, "CSU")

    # Generate notes for each pace and store in dict
    for pace in ["slow", "average", "fast"]:
        result = run_notes_chain(
            student_id    = "student_1019",
            topic_map     = chunks_topic,
            current_topic = topic,
            weak_topics   = [],
            strong_topics = [],
            course        = "CSU",
            learning_pace = pace,
            llm           = llm,
            embedder      = embedder,
            store         = store,
        )
        pdf_path = result.content

        key = str(i).zfill(3)
        json_eval_dataset[key] = {
            "topic_title": topic,
            "learner_pace": pace,
            "notes_file": pdf_path.removeprefix("data/notes/"),
        }
        i += 1



PDF page count: 3
Processing: ADDER CIRCUIT.pdf


INFO:unstructured-client:split_pdf event=plan_created operation_id=da4180d7-ba3b-4f30-9a6a-d2084b656c64 filename=ADDER CIRCUIT.pdf strategy=hi_res page_range=1-3 page_count=3 split_size=2 chunk_count=2 concurrency=5 allow_failed=False cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset


INFO:httpx:HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO:unstructured-client:split_pdf event=batch_start operation_id=da4180d7-ba3b-4f30-9a6a-d2084b656c64 chunk_count=2 concurrency=5 allow_failed=False client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO:httpx:HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO:unstructured-client:split_pdf event=batch_complete operation_id=da4180d7-ba3b-4f30-9a6a-d2084b656c64 chunk_count=2 success_count=2 failure_count=0 transport_failure_count=0 elapsed_ms=9938 allow_failed=False
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5


  → Parsed into 15 document(s)
Loading embedding model: BAAI/bge-small-en-v1.5 (device=cpu)


INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1

...

INFO:google_genai.models:AFC is enabled with max remote calls: 10.



[notes_chain] Processing section 23/23: 'Further tasks'


INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'Further tasks' condensed.


RuntimeError: PDF construction failed for student 'student_1019', topic 'Week 2- Algorithm Correctness and Mathematical Induction'. | Error: 
paragraph text '<para><i> <b>Analysis of EVENSUM(k + 1):</b> We need to understand what the recursive EVENSUM(k + 1) function actually computes. Based on the recursive definition (which is not provided in this specific snippet but implied by the context of recursive algorithms), EVENSUM(k+1) would typically call EVENSUM(k) and add the next term in the sequence. If the sequence of even numbers being summed is 2, 4, 6, ..., then the k-th term is 2k. The sum of the first k even numbers is $\\sum<i>{i=1}^{k} 2i$. The formula given in the claim is $\\sum</i>{i=1}^{n} 2i - 2$. This suggests that the algorithm is summing terms of the form $2i-2$. Let\'s assume the recursive step for EVENSUM(n) is something like: <font name="Courier">EVENSUM(n) = EVENSUM(n-1) + (2</i>n - 2)</font> for n &gt; 1, and EVENSUM(1) = 0.</para>' caused exception Parse error: sa

In [4]:
files = ['Week 2- Algorithm Correctness and Mathematical Induction.pdf', 'Week 3- Algorithm Efficiency and Asymptotic notation.pdf', 'Week 4- Sorting Algorithms I.pdf']

# Worked on last 3
# Generate each note and build eval dataset
i = 43
for file in files:
    # Run ingestion
    topic = file.removesuffix(".pdf")
    docs = process_and_load_file(f"../pdfs/{file}")
    embedder = get_embedding_model()
    vectors = generate_embeddings(docs, embedder)
    store = get_vector_store(student_id="1019", embedder=embedder)
    add_documents_to_chroma(store, vectors, docs, False, "CSU", topic, topic)

    # Run topic-wide retrieval
    chunks_topic = get_topic_chunks(store, topic, "CSU")

    # Generate notes for each pace and store in dict
    for pace in ["slow", "average", "fast"]:
        result = run_notes_chain(
            student_id    = "student_1019",
            topic_map     = chunks_topic,
            current_topic = topic,
            weak_topics   = [],
            strong_topics = [],
            course        = "CSU",
            learning_pace = pace,
            llm           = llm,
            embedder      = embedder,
            store         = store,
        )
        pdf_path = result.content

        key = str(i).zfill(3)
        json_eval_dataset[key] = {
            "topic_title": topic,
            "learner_pace": pace,
            "notes_file": pdf_path.removeprefix("data/notes/"),
        }
        i += 1

PDF page count: 28
Processing: Week 2- Algorithm Correctness and Mathematical Induction.pdf


INFO:unstructured-client:split_pdf event=plan_created operation_id=d35d06e8-0f02-407f-b9d2-6eed94a3f228 filename=Week 2- Algorithm Correctness and Mathematical Induction.pdf strategy=hi_res page_range=1-28 page_count=28 split_size=6 chunk_count=5 concurrency=5 allow_failed=False cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset
INFO:httpx:HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO:unstructured-client:split_pdf event=batch_start operation_id=d35d06e8-0f02-407f-b9d2-6eed94a3f228 chunk_count=5 concurrency=5 allow_failed=False client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO:httpx:HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https:/

  → Parsed into 108 document(s)
Loading embedding model: BAAI/bge-small-en-v1.5 (device=cpu)


INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[notes_chain] ✓ Section 'Insertion Sort is used when' condensed.
[notes_chain] ⏳ Throttle: sleeping 5s before next section.


INFO:google_genai.models:AFC is enabled with max remote calls: 10.



[notes_chain] Processing section 15/15: 'Summary of complexities of the elementary sorting '


INFO:httpx:HTTP Request: POST https://us-central1-aiplatform.googleapis.com/v1beta1/projects/denseless/locations/us-central1/publishers/google/models/gemini-2.5-flash-lite:generateContent "HTTP/1.1 200 OK"


[notes_chain] ✓ Section 'Summary of complexities of the elementary sorting ' condensed.
[notes_chain] PDF built: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Week_4-_Sorting_Algorithms_I_20260525_104918.pdf

[notes_chain] ═══ Notes generation complete ═══
[notes_chain] PDF: c:\Users\USER\Documents\AI projects\DenseLess\data\notes\student_1019_Week_4-_Sorting_Algorithms_I_20260525_104918.pdf
[notes_chain] Sections completed: 15/15
[notes_chain] Total tokens — in: 13,819 | out: 2,547 | total: 16,366
  [token_service] Deducted 16,366 tokens — student: 'student_1019' | chain: run_notes_chain | in: 13,819 | out: 2,547 | remaining: 7,686,517
  [token_guard] ✓ Tokens deducted — student: student_1019 | consumed: 16366 | remaining: 7686517


In [5]:
# Export all information as a json file
output_path = Path(os.getcwd()) / "rag" / "eval_data" / "csu_eval_data.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(json_eval_dataset, f, indent=4, ensure_ascii=False)

print(f"Successfully completed. Exported to {output_path}")

Successfully completed. Exported to c:\Users\USER\Documents\AI projects\DenseLess\app\agent\rag\eval_data\csu_eval_data.json
